# Orchestration & Automation with Databricks Workflows

Multi-task jobs, dependencies, scheduling, retries, and repair runs — the essentials of pipeline orchestration.

| Training Block | Duration | Type |
|---|---|---|
| Orchestration & Automation — Demo | 25 min | Demo |

**Prerequisites:** 03 — Lakeflow Pipelines (understanding of pipeline concepts)

## Learning Objectives

After completing this module you will be able to:

- **Create** multi-task Databricks Workflows with task dependencies
- **Configure** scheduling patterns (CRON expressions)
- **Apply** retry policies and understand repair runs
- **Use** `dbutils.widgets` and `dbutils.jobs.taskValues` for parameterization
- **Monitor** job execution via UI and system tables

## Introduction to Lakeflow Jobs

**Lakeflow Jobs** (formerly Databricks Jobs) is a fully managed orchestration service that lets you define **multi-task workflows** as directed acyclic graphs (DAGs). Each task can be a notebook, pipeline, SQL query, Python script, or even a conditional branch.

A Job run follows this lifecycle:

```
TRIGGER → QUEUED → RUNNING → [RETRY if transient error] → SUCCESS / FAILED
                                                            ↓
                                                      REPAIR RUN (re-run failed + downstream only)
```

| Scenario | Solution |
|----------|----------|
| ETL pipeline with multiple steps | Multi-task Job |
| Daily report at fixed time | Scheduled Job (CRON) |
| Reaction to new files | File Arrival Trigger |
| Reaction to upstream table update | Table Update Trigger |
| ML training pipeline | Job with notebook tasks |
| Run Lakeflow Pipeline | Job with Pipeline task |
| Conditional branching | If/Else task |
| Parameterized batch runs | For Each task |

---

| Feature | Jobs | Lakeflow Pipelines |

|---------|------|-----|**Best Practice**: Use Lakeflow Pipelines for ETL transformations, Jobs for orchestrating Pipelines + other tasks (ML, reports, notifications).

| Orchestration | General-purpose (any task type) | ETL-focused (SQL/Python) |

| Dependencies | Manual configuration (DAG) | Automatic (lineage-based) || Flexibility | High | Opinionated |

| Data Quality | Custom code | Built-in expectations || Compute | Per-task cluster or Serverless | Shared pipeline cluster |
| Error Handling | Retry + Repair Runs | Auto-retry within pipeline |

### Task Types in Lakeflow Jobs

Overview of all available task types in Lakeflow Jobs and when to use each one.

| Task Type | Description | Use Case |
|-----------|-------------|----------|
| **Notebook** | Run a Databricks notebook | ETL logic, ML training |
| **Pipeline** | Run a Lakeflow Declarative Pipeline | Streaming/batch pipelines |
| **Python Script** | Run a `.py` file from Workspace/Repos | Utility scripts, data validation |
| **SQL** | Run a SQL query or file | DDL, reporting queries, CTAS |
| **JAR** | Run a Java/Scala JAR | Legacy Spark jobs |
| **Spark Submit** | Submit a Spark application | Custom Spark apps, external JARs |
| **If/Else Condition** | Branch based on task value or parameter | Conditional workflows |
| **For Each** | Iterate over a JSON array | Parameterized batch runs (e.g., per-region) |

#### Repair Runs

When a Job fails mid-execution, you don't have to re-run the entire pipeline. **Repair Runs** re-execute only the **failed tasks and their downstream dependents**, skipping all previously successful tasks.

> **Pro Tip:** If/Else conditions evaluate `{{tasks.<task_key>.result_state}}` or task values. For Each iterates over a JSON array and runs nested tasks for each element — great for multi-region or multi-table patterns.

```

Run #1:  validate ✅ → transform ❌ → report (skipped)```

Repair:  validate (skipped ✅) → transform 🔄 → report 🔄databricks jobs repair-run <run-id> --rerun-tasks transform,report

```# Databricks CLI

```bash

This saves significant compute time and cost in large pipelines. You can trigger a Repair Run from the UI (Runs tab → Repair) or via API:

## Preparing Notebooks for Job

Below are 3 simple notebooks that we'll use in the demo.

**Instructions**: 
1. Create folder `/Workspace/Users/<your-email>/jobs_demo/`
2. Copy each of the following code snippets to a separate notebook

---

### Task 1: Validate Source

Validates row count against `min_rows` threshold. Returns status + count via `dbutils.notebook.exit()`.

In [0]:
# TASK 1: Validate Source Data
# Copy this code to notebook: task_01_validate

# Parameters from Job
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "bronze.bronze_orders", "Source Table")
dbutils.widgets.text("min_rows", "100")

catalog = dbutils.widgets.get("catalog")
source_table = dbutils.widgets.get("source_table")
min_rows = int(dbutils.widgets.get("min_rows"))

full_table = f"{catalog}.{source_table}" if catalog else source_table

# Validation
df = spark.table(full_table)
row_count = df.count()

if row_count < min_rows:
    raise Exception(f"Validation FAILED: {row_count} rows < {min_rows} minimum")

# Return result to next task
import json
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "source_table": full_table,
    "row_count": row_count
}))


### Task 2: Transform Data

Reads previous task result via `taskValues`, applies transformations (duration, cost per mile). Returns row count.

**Task Values — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Set value (upstream task) | `dbutils.jobs.taskValues.set(key="row_count", value=1000)` |
| Get value (downstream task) | `val = dbutils.jobs.taskValues.get(taskKey="Task_1", key="row_count")` |
| Supported types | `str`, `int`, `float`, `bool`, JSON-serializable `dict`/`list` |
| Max value size | 48 KB per key |


In [0]:
# TASK 2: Transform Data (Bronze → Silver)
# Copy this code to notebook: task_02_transform

from pyspark.sql.functions import *
import json
from datetime import date

# Parameters
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "bronze.bronze_orders", "Source Table")
dbutils.widgets.text("run_date", "")

catalog = dbutils.widgets.get("catalog")
source_table = f'{catalog}.{dbutils.widgets.get("source_table")}' if catalog else dbutils.widgets.get("source_table")
run_date = dbutils.widgets.get("run_date") or str(date.today())

# Get result from previous task (optional)
try:
    prev_result = dbutils.jobs.taskValues.get(
        taskKey="validate",
        key="returnValue",
        default="{}"
    )
    prev_data = json.loads(prev_result)
    print(f"Previous task result: {prev_data}")
except:
    print("Running standalone (no previous task)")

# Transformation: compute net amounts
print(f"Transforming: {source_table}")

df = spark.table(source_table)

df_transformed = (
    df
    .withColumn("gross_amount", col("quantity") * col("unit_price"))
    .withColumn("discount_amount",
                round(col("quantity") * col("unit_price") * col("discount_percent") / 100, 2))
    .withColumn("net_amount",
                round(col("quantity") * col("unit_price") * (1 - col("discount_percent") / 100), 2))
    .withColumn("order_date", col("order_datetime").cast("date"))
    .withColumn("processing_date", lit(run_date))
    .filter(col("order_id").isNotNull())
    .filter(col("quantity") > 0)
)

row_count = df_transformed.count()
print(f"Transformed {row_count:,} rows")

df_transformed.select(
    "order_id", "customer_id", "quantity", "unit_price",
    "gross_amount", "discount_amount", "net_amount"
).show(5)

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "rows_transformed": row_count
}))


### Task 3: Generate Report

Aggregates metrics (total trips, revenue, avg fare/distance). Prints summary report.

In [0]:
# TASK 3: Generate Sales Report
# Copy this code to notebook: task_03_report

from pyspark.sql.functions import *
import json
from datetime import datetime

# Parameters
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("source_table", "silver.silver_orders", "Source Table")

catalog = dbutils.widgets.get("catalog")
source_table = f'{catalog}.{dbutils.widgets.get("source_table")}' if catalog else dbutils.widgets.get("source_table")

# Aggregations
df = spark.table(source_table)

report = df.agg(
    count("*").alias("total_orders"),
    countDistinct("customer_id").alias("unique_customers"),
    round(sum("net_amount"), 2).alias("total_revenue"),
    round(avg("net_amount"), 2).alias("avg_order_value"),
    round(max("net_amount"), 2).alias("max_order_value"),
    round(avg("discount_percent"), 1).alias("avg_discount_pct")
).collect()[0]

# Display report
print("\n" + "=" * 50)
print("  RETAILHUB — DAILY SALES REPORT")
print("=" * 50)
print(f"  Total Orders:      {report.total_orders:,}")
print(f"  Unique Customers:  {report.unique_customers:,}")
print(f"  Total Revenue:     ${report.total_revenue:,.2f}")
print(f"  Avg Order Value:   ${report.avg_order_value:.2f}")
print(f"  Max Order Value:   ${report.max_order_value:.2f}")
print(f"  Avg Discount:      {report.avg_discount_pct}%")
print("=" * 50)
print(f"  Generated at:      {datetime.now()}")
print("=" * 50 + "\n")

# Return result
dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "total_orders": report.total_orders,
    "total_revenue": float(report.total_revenue)
}))


### [UI DEMO] Creating Multi-task Job

**Step 1: Create new Job**
- [ ] Workflows → Create Job
- [ ] Name: `Demo_ETL_Pipeline`

<img src="../../../assets/images/93c107ca21a54aab98249cf47db0337d.png" width="800">


- [ ] Cluster Job: Create new cluster job

<img src="../../../assets/images/a967557a143a40c0ac7ed26ce469866a.png" width="800">

**Step 2: Add Task 1 (Validate)**
- [ ] Task name: `validate`
- [ ] Type: Notebook
- [ ] Path: `/Workspace/.../task_01_validate`
- [ ] Cluster: Serverless or new Job Cluster
- [ ] Parameters: `source_table` = `${CATALOG}.bronze.bronze_orders`

<img src="../../../assets/images/214b868309344df3a6e81f3cc2a84c13.png" width="800">

**Step 3: Add Task 2 (Transform)**
- [ ] Task name: `transform`
- [ ] Depends on: `validate`
- [ ] Path: `/Workspace/.../task_02_transform`
- [ ] Parameters: `source_table` = `${CATALOG}.bronze.bronze_orders`

**Step 4: Add Task 3 (Report)**
- [ ] Task name: `report`
- [ ] Depends on: `transform`
- [ ] Path: `/Workspace/.../task_03_report`

<img src="../../../assets/images/a3cc387f44de4247bf275bdbd38efb84.png" width="800">

**Step 5: Run Job**
- [ ] Run now
- [ ] Show: DAG visualization
- [ ] Show: Task logs and output

---

## [UI DEMO 2] Medallion Pipeline Job — Ready-to-Use Config

We already have 6 production-ready notebooks in `materials/medallion/` that implement a full Medallion pipeline:

| Layer | Notebook | Input | Output |
|-------|----------|-------|--------|
| **Bronze** | `bronze_customers` | CSV files | `bronze.bronze_customers` |
| **Bronze** | `bronze_orders` | JSON files | `bronze.bronze_orders` |
| **Silver** | `silver_customers` | bronze_customers | `silver.silver_customers` |
| **Silver** | `silver_orders_cleaned` | bronze_orders | `silver.silver_orders_cleaned` |
| **Gold** | `gold_customer_orders_summary` | silver_customers + silver_orders | `gold.gold_customer_orders_summary` |
| **Gold** | `gold_daily_orders` | silver_orders | `gold.gold_daily_orders` |

**DAG Structure:**
```
bronze_customers ──→ silver_customers ──────→ gold_customer_orders_summary
bronze_orders ────→ silver_orders_cleaned ─┤
 └→ gold_daily_orders
```

**How to deploy:** Copy the JSON config below → Workflows → Create Job → switch to JSON editor (or use Databricks CLI: `databricks jobs create --json @job.json`).

---

### Declarative Automation Bundles (DABs) — YAML Configuration

Below is the **Declarative Automation Bundles** (formerly Databricks Asset Bundles / DABs) YAML definition for the Medallion pipeline Job. This was exported from a working Databricks deployment.

> **Before deploying:** Update `notebook_path`, `existing_cluster_id`, and `parameters` to match your environment.

```yaml
resources:
  jobs:
    job_demo_2:
      name: job_demo_medalion
      tasks:
        - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/bronze_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_customer
          depends_on:
            - task_key: bronze_customer
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_customers
            source: WORKSPACE
          environment_key: Default
        - task_key: silver_orders_cleaned
          depends_on:
            - task_key: bronze_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/silver_orders_cleaned
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_daily_orders
          depends_on:
            - task_key: silver_orders_cleaned
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_daily_orders
            source: WORKSPACE
          environment_key: Default
        - task_key: gold_customer_orders_summary
          depends_on:
            - task_key: silver_customer
            - task_key: gold_daily_orders
          notebook_task:
            notebook_path: /Workspace/Users/<your_user>/repo/databricks-lakehouse-transformation/materials/medallion/gold_customer_orders_summary
            source: WORKSPACE
          environment_key: Default
      queue:
        enabled: true
      parameters:
        - name: catalog
          default: retailhub_trainer
        - name: source_path
          default: /Volumes/retailhub_trainer/default/datasets
      environments:
        - environment_key: Default
          spec:
            environment_version: "5"
      performance_target: PERFORMANCE_OPTIMIZED

```

| Element | Description |
|---------|-------------|
| `existing_cluster_id` | Use existing all-purpose cluster (for demo) or replace with `job_cluster_key` for production |
| `depends_on` | Defines task dependencies — creates the DAG |
| `parameters` | Job-level parameters passed to all notebooks via `dbutils.widgets` |
| `queue.enabled` | Queues runs if max concurrent reached |
| `source: WORKSPACE` | Notebook is in Workspace (not Repos) |

### Deploy Checklist

**Option A — JSON Editor (UI):**
1. [ ] Workflows → Create Job
2. [ ] Click ` Edit as JSON` (top-right)
3. [ ] Paste the JSON above (replace placeholders)
4. [ ] Save → Run now

**Option B — Databricks CLI:**
```bash
# Save JSON to file, then:
databricks jobs create --json @medallion_pipeline_job.json
```

**After deployment — show participants:**
- [ ] DAG visualization (fan-out at Bronze, fan-in at Gold)
- [ ] Task-level parameters (catalog, schema, source_path)
- [ ] Trigger: set to PAUSED — run manually for demo
- [ ] Run → observe sequential layer execution (Bronze → Silver → Gold)
- [ ] Show Repair Run: intentionally fail one task → repair reruns only failed + downstream

---

## [UI DEMO] Triggers and Schedule

Lakeflow Jobs supports multiple trigger types. A Job can have **multiple triggers** simultaneously (e.g., scheduled + file arrival). Each trigger creates an independent run.

---

| Trigger Type | Behavior | Example | Max Concurrency |
|---|---|---|---|
| **Scheduled (CRON)** | Fixed schedule using Quartz CRON | `0 0 2 * * ?` — daily at 2:00 | Configurable |
| **File arrival** | Fires when new files land in a UC Volume or cloud path | New CSV in `/Volumes/.../landing/` | 1 (default) |
| **Table update** | Fires when an upstream Delta table is updated | `catalog.schema.bronze_orders` modified | 1 (default) |
| **Continuous** | Restarts immediately after completion | Streaming-like always-on | 1 |
| **Manual** | On-demand via UI, CLI, or API | Testing, ad-hoc runs | Configurable |


#### CRON Syntax (Quartz format)> **Pro Tip:** `?` means "no specific value" — use it in either day-of-month OR day-of-week (not both). File arrival triggers check for new files every ~60 seconds.



Databricks uses **Quartz CRON** with 6-7 fields: `second minute hour day-of-month month day-of-week [year]`| Day of week | 1-7 or SUN-SAT | `, - * ? / L #` |

| Month | 1-12 or JAN-DEC | `, - * /` |

| Field | Allowed Values | Special Characters || Day of month | 1-31 | `, - * ? / L W` |

|-------|---------------|--------------------|| Hour | 0-23 | `, - * /` |

| Second | 0-59 | `, - * /` || Minute | 0-59 | `, - * /` |


### Trigger Configuration Checklist

Step-by-step instructor checklist for demonstrating trigger options in the Lakeflow Jobs UI.

**Trigger Options** (Triggers tab):

| Trigger Type | Usage | Example |
|--------------|-------|---------|
| **Scheduled** | Fixed schedule | Daily at 2:00 |
| **File arrival** | Reaction to new files | New file in `/landing/` |
| **Continuous** | Continuous processing | Streaming-like |
| **Manual** | On-demand | Testing |

<img src="../../../assets/images/6e23746ce063491fa8afb3dea6268a1d.png" width="800">


**Demo: Scheduled Trigger**
- [ ] Add trigger → Scheduled
- [ ] Cron expression: `0 0 2 * * ?` (daily at 2:00)
- [ ] Timezone: `Europe/Warsaw`
- [ ] Show: Preview next runs


<img src="../../../assets/images/cf3cfc85162a466eb77e15e20df5c15c.png" width="800">

**Demo: File Arrival Trigger** (optional)
- [ ] Add trigger → File arrival
- [ ] URL: Unity Catalog Volume path
- [ ] Min files: 1
### Useful CRON Expressions

Common CRON patterns for scheduling Lakeflow Jobs at various intervals.

```
0 0 2 * * ? # Daily at 2:00
0 0 * * * ? # Every hour
0 0 9 ? * MON-FRI # Mon-Fri at 9:00
0 0 0 1 * ? # First day of month
0 */15 * * * ? # Every 15 minutes
```

---

## [UI DEMO] Options, Retry and Alerting

Lakeflow Jobs provides granular control over failure handling at both **task** and **job** levels.

### Task-Level Options

| Option | Description | Default |
|--------|-------------|---------|
| **Timeout** | Max execution time before task is killed | None (unlimited) |
| **Retries** | Number of retry attempts on failure | 0 |
| **Retry delay** | Wait time between retries (seconds) | 0 |
| **Depends on** | Which tasks must complete first | None |

### Job-Level Options

---

| Option | Description | Default |

|--------|-------------|---------|> **Pro Tip:** Set retries only for tasks that call external APIs or depend on transient resources. For data quality tasks, use `0` retries and rely on alerting to notify the team.

| **Max concurrent runs** | How many runs can execute in parallel | 1 |

| **Job timeout** | Max total duration for entire Job | None |Webhook setup: **Workspace Settings → Notification Destinations → Add** (Slack incoming webhook URL or Teams connector URL).

| **Queue** | Queue new runs when max concurrency reached | Disabled |

| **On duration exceeded** | Email, Webhook |

### Retry Decision Matrix| **On failure** | Email, Webhook |

| **On success** | Email, Webhook |

| Scenario | Retry? | Why || **On start** | Email, Webhook (Slack/Teams) |

|----------|--------|-----||-------|---------------------|

| Network timeout | ✅ Yes | Transient error — likely resolves on retry || Event | Notification Options |

| API rate limit (429) | ✅ Yes | Back-off and retry |

| Cluster startup failure | ✅ Yes | Transient infrastructure issue |Configure alerts on Job/task events:

| Data quality / assertion error | ❌ No | Retry won't fix data — fix source first |

| Code bug / syntax error | ❌ No | Retry won't fix code — fix and repair |### Notification Destinations

| Permission denied | ❌ No | Configuration issue — fix RBAC |

## Demo: Widgets and Parameters

Databricks Widgets allow you to parameterize notebooks — making them reusable across environments (dev/staging/prod), tables, and date ranges. Widgets create input controls at the top of the notebook.

When a notebook runs as a **Job task**, widget values are set via the task's **Parameters** configuration. When run interactively, the user can modify values via the UI.

---

#### Dynamic Job References

| `{{job.start_time}}` | Full timestamp (ms) | `1713254400000` |

In Job task parameters, you can use built-in **dynamic value references** that resolve at runtime:| `{{job.start_time.iso_date}}` | Run date (ISO) | `2026-04-16` |

| `{{task.name}}` | Current task name | `transform` |

| Reference | Resolves To | Example Value || `{{run.id}}` | Current run ID | `987654321` |

|-----------|-------------|---------------|| `{{job.id}}` | Job ID | `123456789` |

**Databricks Widgets — Syntax Reference**

| Widget Type | Syntax |
|-------------|--------|
| Text input | `dbutils.widgets.text("name", "default", "Label")` |
| Dropdown | `dbutils.widgets.dropdown("name", "default", ["opt1", "opt2"], "Label")` |
| Multi-select | `dbutils.widgets.multiselect("name", "default", ["opt1", "opt2"], "Label")` |
| Combobox | `dbutils.widgets.combobox("name", "default", ["opt1", "opt2"], "Label")` |
| Get value | `val = dbutils.widgets.get("name")` |
| Remove all | `dbutils.widgets.removeAll()` |


In [0]:
# Widget types

# Text - any text
dbutils.widgets.text("environment", "dev", "Environment")

# Dropdown - select from list
dbutils.widgets.dropdown("region", "EU", ["EU", "US", "APAC"], "Region")

# Combobox - dropdown with typing option
dbutils.widgets.combobox("table", "orders", ["orders", "customers", "products"], "Table")

# Multiselect - multiple selection
dbutils.widgets.multiselect("columns", "id", ["id", "name", "date", "amount"], "Columns")

In [0]:
# Getting values
environment = dbutils.widgets.get("environment")
region = dbutils.widgets.get("region")
table = dbutils.widgets.get("table")
columns = dbutils.widgets.get("columns")  # returns comma-separated string

print(f"Environment: {environment}")
print(f"Region: {region}")
print(f"Table: {table}")
print(f"Columns: {columns}")

In [0]:
# Dynamic parameters in Job
# These values are available when notebook is run as a task in Job

dynamic_params = {
    "{{job.start_time.iso_date}}": "Run date (YYYY-MM-DD)",
    "{{job.start_time}}": "Full timestamp",
    "{{job.id}}": "Job ID",
    "{{run.id}}": "Current run ID",
    "{{task.name}}": "Current task name"
}

for param, description in dynamic_params.items():
    print(f"{param:35} -> {description}")

In [0]:
# Widget cleanup
dbutils.widgets.removeAll()

### Orchestrating Notebooks — dbutils.notebook.run()

While Lakeflow Jobs is the production-grade orchestrator, `dbutils.notebook.run()` lets you call one notebook from another programmatically — useful for lightweight pipelines and testing.

```python
result = dbutils.notebook.run(
    path             = "/path/to/child_notebook",
    timeout_seconds  = 300,
    arguments        = {"env": "prod", "min_rows": "500"}
)
print(result)   # child calls dbutils.notebook.exit("OK: 1234 rows")
```

#### Task Values — passing data between Job tasks

In a multi-task Job you can set and retrieve task-scoped values:

```python
# In task A — set a value
dbutils.jobs.taskValues.set("row_count", 1234)

# In task B (downstream) — read the value
count = dbutils.jobs.taskValues.get("task_A", "row_count", default=0)
```

| Method | Scope | Use Case |
|---|---|---|
| `dbutils.notebook.run()` | Notebook-to-notebook | Quick orchestration, child notebook pattern |
| `dbutils.jobs.taskValues` | Task-to-task in a Job | Pass computed values between Job tasks |

> **Pro Tip:** `dbutils.notebook.run()` is synchronous — it blocks until the child finishes or the timeout is reached. For parallel execution use Lakeflow Jobs task dependencies.

---

In [ ]:
# dbutils.notebook.run() — programmatic notebook orchestration
# (Uncomment to use in production; the path below is illustrative)

# result = dbutils.notebook.run(
#     "/path/to/validation_notebook",
#     timeout_seconds = 120,
#     arguments       = {
#         "env"     : dbutils.widgets.get("environment"),
#         "min_rows": "100"
#     }
# )
# print(f"Child notebook returned: {result}")

# ── dbutils.jobs.taskValues — passing data between Job tasks ──────────────
# In Task A:
# dbutils.jobs.taskValues.set("validated_rows", 9999)

# In Task B:
# row_count = dbutils.jobs.taskValues.get("task_a", "validated_rows", default=0)

# Signature check
print("dbutils.notebook.run  →", dbutils.notebook.run.__doc__[:80] if hasattr(dbutils.notebook.run,'__doc__') and dbutils.notebook.run.__doc__ else "available in Databricks runtime")
print("dbutils.jobs.taskValues →", "set(key, value) / get(taskKey, key, default)")

## Bonus: Monitoring via System Tables

Databricks provides **System Tables** — read-only Delta tables in the `system` catalog that record platform activity. For Job monitoring, the key tables are:

| System Table | Content | Retention |

|---|---|---|---

| `system.lakeflow.job_run_timeline` | Job run history (start, end, result, duration) | 365 days |

| `system.lakeflow.job_task_run_timeline` | Task-level run details within jobs | 365 days |> **Pro Tip:** System tables are available only with **Unity Catalog enabled**. They are excellent for building observability dashboards, SLA tracking, and cost attribution.

| `system.lakeflow.cluster_timeline` | Cluster usage by jobs | 365 days |

| `system.compute.warehouse_events` | SQL Warehouse events | 365 days || `trigger_type` | STRING | `PERIODIC`, `FILE_ARRIVAL`, `MANUAL`, etc. |

| `period_end_time` | TIMESTAMP | Run end time |

#### Key Columns in `job_run_timeline`| `period_start_time` | TIMESTAMP | Run start time |

| `result_state` | STRING | `SUCCESS`, `FAILED`, `TIMEDOUT`, `CANCELED` |

| Column | Type | Description || `run_name` | STRING | Job name at time of run |

|--------|------|-------------|| `run_id` | LONG | Run identifier |

| `workspace_id` | LONG | Workspace identifier || `job_id` | LONG | Job identifier |

In [0]:
spark.sql("""
    SELECT 
        *
    FROM system.lakeflow.job_run_timeline
    ORDER BY 1 DESC
    LIMIT 20
""").display()

In [0]:
spark.sql("""
    SELECT 
        DATE(period_start_time) as run_date,
        run_name as job_name,
        ROUND(
            AVG(
                (UNIX_TIMESTAMP(period_end_time) - UNIX_TIMESTAMP(period_start_time)) / 60
            ), 1
        ) as avg_duration_min,
        COUNT(*) as runs
    FROM system.lakeflow.job_run_timeline
    WHERE period_start_time >= current_date() - INTERVAL 14 DAYS
        AND result_state = 'SUCCESS'
    GROUP BY run_date, job_name
    ORDER BY run_date DESC
""").display()

## Serverless Compute for Jobs

Since 2025, Databricks offers **Serverless compute** as the default option for Lakeflow Jobs. Serverless eliminates cluster management — tasks start in seconds with auto-scaled, ephemeral compute.

| Aspect | Classic Clusters | Serverless Compute |
|--------|-----------------|-------------------|
| **Startup time** | 3-10 minutes | Seconds |
| **Cluster management** | Manual sizing (workers, instance types) | Fully automatic |
| **Cost model** | Per-VM (DBU + cloud infra) | Per-DBU only (no infra cost) |
| **Spot instances** | Configurable (cost savings) | Not applicable |
| **Auto-scaling** | Configurable (min/max workers) | Built-in, transparent |
| **Custom libraries** | Full control (init scripts, pip) | Limited (pip only, no init scripts) |
| **GPU support** | Yes | No |
| **Use case** | Long-running, GPU, custom libs, cost-sensitive batch | ETL, SQL, notebooks, triggered jobs |

**How to enable:** In the Job task configuration, select **Serverless** as the compute type instead of specifying a cluster.


#### When to Use Classic Clusters Instead---



| Scenario | Why Classic |> **Pro Tip:** Serverless compute is ideal for **triggered jobs** that need fast response times (file arrival, table update). For cost-sensitive scheduled batch jobs running for hours, classic clusters with spot instances may still be 30-50% cheaper.

|----------|-------------|

| GPU / ML training | Serverless doesn't support GPU instances || Network isolation (Private Link) | Classic clusters support custom VNet/VPC |

| Custom init scripts | Serverless only supports pip packages || Specific instance types needed | Serverless auto-selects instance types |
| Long-running batch (hours) | Spot instances on classic can be significantly cheaper |

## Summary

| Topic | Key Concept | Key Terms |
|---|---|---|
| **Multi-task Jobs** | DAG workflow, task dependencies | Task types, Repair Runs |
| **Triggers** | Scheduled (CRON), File arrival, Continuous | `0 0 2 * * ?` |
| **Options** | Timeout, Retry, Max concurrent runs | Transient vs permanent errors |
| **Alerting** | Email, Webhooks (Slack/Teams) | Notification destinations |
| **Parameters** | Widgets, dynamic values, taskValues | `dbutils.widgets`, `dbutils.jobs.taskValues` |
| **Monitoring** | System tables, success rate, duration | `system.lakeflow.job_run_timeline` |

---

← [03 — Lakeflow Pipelines](03_lakeflow_pipelines_demo.ipynb) | **[ README](../../../README.md)**
